# 小编环境

In [1]:
import sys

print('python 版本：', sys.version.split('|')[0])
#python 版本： 3.11.11 

import polars as pl

print("polars 版本：", pl.__version__)
#polars 版本： 1.37.1

python 版本： 3.11.11 
polars 版本： 1.37.1


# api 函数

- map_elements ：对列中的每个值，传入函数，类似pandas中的map
- map_batches ：整个列全部传入函数，类似pandas中的apply

# map_elements 用法

In [2]:
df = pl.DataFrame(
    {
        "keys": ["a", "a", "b", "b"],
        "values": [10, 7, 1, 23],
    }
)
print(df)

shape: (4, 2)
┌──────┬────────┐
│ keys ┆ values │
│ ---  ┆ ---    │
│ str  ┆ i64    │
╞══════╪════════╡
│ a    ┆ 10     │
│ a    ┆ 7      │
│ b    ┆ 1      │
│ b    ┆ 23     │
└──────┴────────┘


In [3]:
import math 


def my_log(value):
    return math.log(value)  # math.log 应用与每个值


out = df.select(pl.col("values").map_elements(my_log, return_dtype=pl.Float64))
print(out)

C:\Users\admin\AppData\Local\Temp\ipykernel_26576\1223290643.py:8: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("values").map_elements(my_log)
with this one instead:
  + pl.col("values").log()

  out = df.select(pl.col("values").map_elements(my_log, return_dtype=pl.Float64))


shape: (4, 1)
┌──────────┐
│ values   │
│ ---      │
│ f64      │
╞══════════╡
│ 2.302585 │
│ 1.94591  │
│ 0.0      │
│ 3.135494 │
└──────────┘


存在问题：
1. **限于单个项**：只用应用在单个值上面，而不能一次应用到整个列
2. **性能开销**：为每个单独的项调用函数也很慢，所有这些额外的函数调用会增加大量的开销

# map_batches 用法

In [4]:
def diff_from_mean(series):
    # This will be very slow for non-trivial Series, since it's all Python
    # code:
    total = 0
    for value in series:
        total += value
    mean = total / len(series)
    return pl.Series([value - mean for value in series])


# Apply our custom function to a full Series with map_batches():
out = df.select(pl.col("values").map_batches(diff_from_mean, return_dtype=pl.Float64))
print("== select() with UDF ==")
print(out)

# Apply our custom function per group:
print("== group_by() with UDF ==")
out = df.group_by("keys").agg(
    pl.col("values").map_batches(diff_from_mean, return_dtype=pl.Float64)
)
print(out)

== select() with UDF ==
shape: (4, 1)
┌────────┐
│ values │
│ ---    │
│ f64    │
╞════════╡
│ -0.25  │
│ -3.25  │
│ -9.25  │
│ 12.75  │
└────────┘
== group_by() with UDF ==
shape: (2, 2)
┌──────┬───────────────┐
│ keys ┆ values        │
│ ---  ┆ ---           │
│ str  ┆ list[f64]     │
╞══════╪═══════════════╡
│ b    ┆ [-11.0, 11.0] │
│ a    ┆ [1.5, -1.5]   │
└──────┴───────────────┘


# 提升用户自定义函数性能

## numpy 通用函数

纯python实现的自定义函数一般速度都比较慢，要尽量减少代用python实现的方法

可以调用 numpy 中的实现的通用函数/算子，来加速

In [5]:
import numpy as np

out = df.select(pl.col("values").map_batches(np.log, return_dtype=pl.Float64))
print(out)

shape: (4, 1)
┌──────────┐
│ values   │
│ ---      │
│ f64      │
╞══════════╡
│ 2.302585 │
│ 1.94591  │
│ 0.0      │
│ 3.135494 │
└──────────┘


## 通过 Numba 提升自定义函数性能

In [6]:
from numba import guvectorize,int64,float64

@guvectorize([(int64[:], float64[:])], "(n)->(n)")
def diff_from_mean_numba(arr, result):
    total = 0
    for value in arr:
        total += value
    mean = total / len(arr)
    for i, value in enumerate(arr):
        result[i] = value - mean


out = df.select(
    pl.col("values").map_batches(diff_from_mean_numba, return_dtype=pl.Float64)
)
print("== select() with UDF ==")
print(out)

out = df.group_by("keys").agg(
    pl.col("values").map_batches(diff_from_mean_numba, return_dtype=pl.Float64)
)
print("== group_by() with UDF ==")
print(out)

== select() with UDF ==
shape: (4, 1)
┌────────┐
│ values │
│ ---    │
│ f64    │
╞════════╡
│ -0.25  │
│ -3.25  │
│ -9.25  │
│ 12.75  │
└────────┘
== group_by() with UDF ==
shape: (2, 2)
┌──────┬───────────────┐
│ keys ┆ values        │
│ ---  ┆ ---           │
│ str  ┆ list[f64]     │
╞══════╪═══════════════╡
│ a    ┆ [1.5, -1.5]   │
│ b    ┆ [-11.0, 11.0] │
└──────┴───────────────┘


## 注意事项

**加速时，数据缺失是不行的**，在利用numba装饰器`@guvectorize`加速时，要么填充缺失值，要么删除缺失值，否则polars会报错

# 组合多列

In [7]:
# Add two arrays together:
@guvectorize([(int64[:], int64[:], float64[:])], "(n),(n)->(n)")
def add(arr, arr2, result):
    for i in range(len(arr)):
        result[i] = arr[i] + arr2[i]


df3 = pl.DataFrame({"values_1": [1, 2, 3], "values_2": [10, 20, 30]})

out = df3.select(
    # Create a struct that has two columns in it:
    pl.struct(["values_1", "values_2"])
    # Pass the struct to a lambda that then passes the individual columns to
    # the add() function:
    .map_batches(
        lambda combined: add(
            combined.struct.field("values_1"), combined.struct.field("values_2")
        ),
        return_dtype=pl.Float64,
    )
    .alias("add_columns")
)
print(out)

shape: (3, 1)
┌─────────────┐
│ add_columns │
│ ---         │
│ f64         │
╞═════════════╡
│ 11.0        │
│ 22.0        │
│ 33.0        │
└─────────────┘


# 流式计算

可以使用 `map_batches` 的 `is_elementwise=True` 参数将结果流式传输到函数中

设置流式计算，需要确保是针对每个值进行计算，更节省内存

# 返回数据类型

返回数据类型是自动推断的，第一个非空值类型，作为结果类型

python 与 Polars 数据类型映射：
- int -> Int64
- float -> Float64
- bool -> Boolean
- str -> String
- list[tp] -> List[tp]
- dict[str, [tp]] -> struct 
- any -> object  尽量禁止这种情况



可以将 `return_dtype` 参数传递给 `map_batches`